# AutoPost Sprint 5 — Kaggle GPU validation

Run the numbered code cells in order. First publish the local Sprint 5 source to
GitHub; a clone of the older main branch does not contain this implementation.

Notebook settings: Python, Internet ON, GPU enabled (T4 if available). A P100 is
below this backend's compute-capability floor and must not be used for this gate.
Only GPU 0 is used, including when Kaggle supplies two T4s.

Optional upload: Add-ons → Secrets → add `GITHUB_TOKEN` and enable access for this
notebook. Use a token restricted to `DakshUbhadia/AutoPost` with repository
Contents read/write. Keep the notebook private while working. Never paste the
token into a cell, output, Git URL or chat. The token is retrieved only by the
last optional cell, after GPU subprocesses finish.

The dependency input is an unvalidated candidate. If setup/preflight fails, stop
the compute cells and run the packaging/upload cells to preserve the error.
Do not alter pins or tolerances merely to make a check pass.


## Cell 1 — Configuration, GPU check and persistent run folder

Re-running this cell starts a new evidence folder. Temporary environments/caches
are outside `/kaggle/working`; only the run evidence and ZIP are notebook outputs.


In [ ]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY = "DakshUbhadia/AutoPost"
SOURCE_BRANCH = "main"
WORK = Path("/kaggle/working")
SCRATCH = Path("/tmp/autopost-sprint5")
SOURCE = SCRATCH / "AutoPost"
VENV = SCRATCH / "venv"
PYTHON = VENV / "bin/python"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
OUTPUT = WORK / ("sprint5-" + RUN_ID)
OUTPUT.mkdir(parents=True, exist_ok=False)
SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ)
for name in ("GITHUB_TOKEN", "GH_TOKEN", "PIP_EXTRA_INDEX_URL"):
    ENV.pop(name, None)
ENV.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "HF_HOME": str(SCRATCH / "cache/huggingface"),
    "VLLM_CACHE_ROOT": str(SCRATCH / "cache/vllm"),
    "PIP_CACHE_DIR": str(SCRATCH / "cache/pip"),
    "UV_CACHE_DIR": str(SCRATCH / "cache/uv"),
    "UV_PYTHON_INSTALL_DIR": str(SCRATCH / "python"),
    "PIP_INDEX_URL": "https://pypi.org/simple",
    "TOKENIZERS_PARALLELISM": "false",
})
STATE = {"schema": "autopost.kaggle-receipt/1", "run_id": RUN_ID,
         "repository": REPOSITORY, "status": "SETUP", "scientific_execution": False,
         "started_at_utc": datetime.now(timezone.utc).isoformat()}

def save_state():
    (OUTPUT / "notebook-receipt.json").write_text(json.dumps(STATE, indent=2) + "\n")

def run_logged(arguments, label, allowed=(0,), cwd=None):
    arguments = [str(item) for item in arguments]
    print(f"\n--- {label} ---", flush=True)
    folder = cwd or (SOURCE if (SOURCE / ".git").is_dir() else SCRATCH)
    with (OUTPUT / f"{label}.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(arguments, cwd=folder, env=ENV,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace")
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            exit_code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            STATE.update(status="INTERRUPTED", failed_step=label)
            save_state()
            raise
    STATE.setdefault("commands", []).append({"step": label, "argv": arguments, "exit_code": exit_code})
    if exit_code not in allowed:
        STATE.update(status="STEP_FAILED", failed_step=label)
        save_state()
        raise RuntimeError(f"{label} failed with exit code {exit_code}. Preserve {label}.log; run cells 6/7 next.")
    save_state()
    return exit_code

save_state()
run_logged(["nvidia-smi"], "nvidia-smi")
hardware = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version", "--format=csv,noheader"],
    text=True, env=ENV,
)
(OUTPUT / "hardware.csv").write_text(hardware)
rows = list(csv.reader(hardware.splitlines(), skipinitialspace=True))
assert rows, "No visible NVIDIA GPU. Enable GPU in notebook settings."
capability = tuple(int(part) for part in rows[0][1].strip().split("."))
STATE["allocated_gpus"] = rows
STATE["selected_gpu"] = 0
STATE["compute_capability"] = list(capability)
if capability < (7, 5):
    STATE.update(status="BLOCKED", reason="GPU compute capability is below 7.5")
    save_state()
    raise RuntimeError("This GPU cannot pass the current vLLM gate. Select a T4 or newer compatible GPU.")
PROFILE = "execution/dev-qwen06b-local24g.yaml" if capability >= (8, 0) else "execution/dev-qwen06b-t4.yaml"
STATE["execution_profile"] = PROFILE
save_state()
print("Selected profile:", PROFILE)
print("Evidence directory:", OUTPUT)


## Cell 2 — Clone the published source and record the exact commit

This uses the public HTTPS repository URL without a token. It refuses a source
folder with uncommitted edits rather than discarding edits on a rerun.


In [ ]:
public_url = f"https://github.com/{REPOSITORY}.git"
if not SOURCE.exists():
    run_logged(["git", "clone", "--depth", "1", "--branch", SOURCE_BRANCH, public_url, SOURCE], "clone", cwd=SCRATCH)
else:
    assert (SOURCE / ".git").is_dir(), "Source path exists but is not a Git checkout. Use a fresh session."
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=SOURCE, text=True).strip()
    assert remote == public_url, "Unexpected source repository; use a fresh session."
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=SOURCE, text=True).strip()
    assert not dirty, "Source contains edits; preserve them before refreshing this notebook."
    run_logged(["git", "fetch", "--depth", "1", "origin", SOURCE_BRANCH], "fetch-source")
    run_logged(["git", "checkout", "--detach", "FETCH_HEAD"], "checkout-source")

required = ["scripts/probe_dev_model.py", "requirements/small-gpu-dev.in",
            "src/autopost/execution/spike.py", "configs/execution/dev-qwen06b-t4.yaml"]
missing = [name for name in required if not (SOURCE / name).is_file()]
if missing:
    STATE.update(status="BLOCKED", reason="Sprint 5 source has not been published", missing=missing)
    save_state()
    raise RuntimeError("Push the local Sprint 5 implementation to GitHub first. Missing: " + ", ".join(missing))
SOURCE_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True).strip()
STATE["source_commit"] = SOURCE_SHA
STATE["candidate_requirements_sha256"] = hashlib.sha256((SOURCE / "requirements/small-gpu-dev.in").read_bytes()).hexdigest()
(OUTPUT / "source-commit.txt").write_text(SOURCE_SHA + "\n")
save_state()
print("Source commit:", SOURCE_SHA)


## Cell 3 — Create isolated Python 3.13.7 and install the candidate stack

This can download large CUDA wheels. The dependency dry run runs first. If it
fails, keep the resolver log; do not continue to GPU execution or silently change
versions. Notebook Python stays separate from the execution environment.


In [ ]:
run_logged([sys.executable, "-m", "pip", "install", "uv==0.13.0"], "bootstrap-uv")
if not PYTHON.is_file():
    run_logged([sys.executable, "-m", "uv", "venv", "--python", "3.13.7", "--seed", VENV], "create-venv")
version = subprocess.check_output([PYTHON, "-c", "import sys; print(sys.version.split()[0])"], text=True, env=ENV).strip()
assert version == "3.13.7", f"Expected Python 3.13.7; got {version}"
STATE["execution_python"] = version
save_state()
run_logged([PYTHON, "-m", "pip", "install", "setuptools==80.9.0"], "build-tools")
run_logged([PYTHON, "-m", "pip", "install", "--dry-run", "--report", OUTPUT / "dependency-plan.json",
            "--requirement", SOURCE / "requirements/small-gpu-dev.in"], "resolve-candidate")
run_logged([PYTHON, "-m", "pip", "install", "--report", OUTPUT / "dependency-install.json",
            "--requirement", SOURCE / "requirements/small-gpu-dev.in"], "install-candidate")
run_logged([PYTHON, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "--no-build-isolation",
            "--force-reinstall", SOURCE], "install-autopost")
run_logged([PYTHON, "-m", "pip", "check"], "pip-check")
run_logged([PYTHON, "-c", "import autopost; print(autopost.__file__)"], "installed-package")


## Cell 4 — Subprocess fixture and GPU preflight

Exit code 2 from preflight is expected for `READY_NOT_RUN`. Proceed only when
that exact status is present; `BLOCKED` is not a pass.


In [ ]:
run_logged([PYTHON, "-m", "autopost.cli", "execution", "fixture", "--root", OUTPUT / "fixture"], "subprocess-fixture")
CAMPAIGNS = OUTPUT / "campaigns"
run_logged([PYTHON, SOURCE / "scripts/probe_dev_model.py", "--execution", PROFILE,
            "--root", CAMPAIGNS], "gpu-preflight", allowed=(0, 2))
preflight = json.loads((CAMPAIGNS / "latest-report.json").read_text())
print("Preflight:", preflight["status"])
if preflight["status"] != "READY_NOT_RUN":
    STATE.update(status="BLOCKED", preflight_status=preflight["status"],
                 reasons=preflight["capabilities"]["reasons"])
    save_state()
    raise RuntimeError("GPU preflight blocked. Run cells 6/7 to preserve the report and logs.")
STATE.update(status="READY_NOT_RUN")
save_state()


## Cell 5 — Real GPU compatibility campaign

Runs adapter export/reload/generation parity, then one disposable TRL step and
the required importance-sampling metrics checks. Each phase has a 20-minute
deadline. Failure evidence is retained. Do not start Sprint 6 unless the campaign
passes and its report/lock are reviewed.


In [ ]:
assert STATE["status"] == "READY_NOT_RUN", "Complete successful preflight first."
previous_campaign = json.loads((CAMPAIGNS / "latest-report.json").read_text())["campaign"]
STATE.update(status="RUNNING")
save_state()
exit_code = run_logged([PYTHON, SOURCE / "scripts/probe_dev_model.py", "--execution", PROFILE,
                        "--root", CAMPAIGNS, "--run-gpu"], "gpu-probe", allowed=(0, 1, 2))
report_path = CAMPAIGNS / "latest-report.json"
report = json.loads(report_path.read_text()) if report_path.is_file() else {}
fresh = report.get("campaign") is not None and report["campaign"] != previous_campaign
passed = exit_code == 0 and fresh and report.get("status") == "PASS"
status = "PASS" if passed else (report.get("status", "RUN_CRASHED") if fresh else "RUN_CRASHED")
if not passed and status == "PASS":
    status = "RUN_CRASHED"
STATE.update(status=status, probe_exit_code=exit_code, report_is_fresh=fresh,
             finished_at_utc=datetime.now(timezone.utc).isoformat())
if fresh:
    STATE.update(campaign=report["campaign"], stack_id=report["capabilities"]["stack_id"],
                 model_spec_id=report["model_spec_id"], execution_profile_id=report["execution_profile_id"])
save_state()
print("\nFINAL STATUS:", status)
if fresh:
    print(json.dumps({key: report.get(key) for key in ("status", "runtime", "checks")}, indent=2))
if not passed:
    for log in sorted(CAMPAIGNS.rglob("*.stderr.log")):
        print(f"\n--- last 40 lines: {log.relative_to(OUTPUT)} ---")
        print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))
print("Run cell 6 even when validation failed.")


## Cell 6 — Package all evidence for download / notebook outputs

Run this after success or any earlier failure. The full ZIP contains the campaign,
including adapter artifacts; the optional GitHub upload sends only selected small
text evidence and hashes. Download the ZIP from Kaggle Outputs and save a notebook
version with outputs before ending the session. Do not rely on temporary caches.


In [ ]:
def package_evidence():
    save_state()
    inventory = []
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and not path.is_symlink() and path.name != "artifact-manifest.json":
            inventory.append({"path": path.relative_to(OUTPUT).as_posix(),
                              "size_bytes": path.stat().st_size,
                              "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    (OUTPUT / "artifact-manifest.json").write_text(json.dumps(inventory, indent=2) + "\n")
    archive = Path(shutil.make_archive(str(WORK / ("sprint5-evidence-" + RUN_ID)), "zip", root_dir=OUTPUT))
    print("Status:", STATE["status"])
    print("Download from Kaggle Outputs:", archive.name)
    print("ZIP size (MiB):", round(archive.stat().st_size / 1024**2, 2))
    return archive

ARCHIVE = package_evidence()
from IPython.display import FileLink, display
display(FileLink(str(ARCHIVE)))


## Cell 7 — Optional: publish small evidence to a new GitHub branch

Set `UPLOAD_TO_GITHUB = True` only if you want this cell to create a results
branch in your repository. `GITHUB_TOKEN` must be enabled in Kaggle Secrets.
It needs Contents read/write; no workflow, administration or pull-request write
permission is needed for this cell. The API token is used only in HTTPS headers,
never in subprocess arguments, environment variables, Git config or files.

The commit is based on the exact tested source SHA. It adds reports/log excerpts
under `docs/gpu-runs/<run-id>/`, not model weights, databases, caches or a venv.
Both failed and successful reports are useful. It does not change main or mark
the GPU environment ready. Review the branch and reports before merging.


In [ ]:
UPLOAD_TO_GITHUB = False  # Change to True to publish the evidence branch.

def upload_results():
    import base64
    import requests
    from kaggle_secrets import UserSecretsClient

    assert STATE.get("source_commit"), "No source commit was checked out; share the ZIP instead."
    assert STATE["status"] != "RUNNING", "Finish/stop the GPU campaign and package its result first."
    allowed_names = {
        "notebook-receipt.json", "artifact-manifest.json", "hardware.csv", "source-commit.txt",
        "latest-report.json", "report.json", "config.json", "identity.json", "adapter.json", "trainer.json",
        "trainer-log-history.json", "raw-generations.json", "raw-trainer-completions.json",
        "environment.lock", "environment.observed.lock", "pip-check.txt",
    }
    evidence = {}
    for path in sorted(OUTPUT.rglob("*")):
        if not path.is_file() or path.is_symlink():
            continue
        if path.name not in allowed_names and path.suffix != ".log":
            continue
        relative = path.relative_to(OUTPUT).as_posix()
        data = path.read_bytes()
        if len(data) > 1024**2:
            relative += ".tail.txt"
            data = b"[Tail only: full file and its hash are in the saved evidence ZIP.]\n" + data[-1024**2:]
        if path.suffix == ".log":
            relative += ".txt"
        evidence[f"docs/gpu-runs/{RUN_ID}/{relative}"] = data
    assert evidence, "No evidence was packaged. Run cell 6 first."

    branch = "gpu/sprint5-" + RUN_ID
    prefix = f"https://api.github.com/repos/{REPOSITORY}"
    with requests.Session() as api:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        api.headers.update({"Authorization": "Bearer " + token, "Accept": "application/vnd.github+json",
                            "X-GitHub-Api-Version": "2022-11-28"})
        del token

        def request(method, route, payload=None):
            response = api.request(method, prefix + route, json=payload, timeout=60)
            if not response.ok:
                raise RuntimeError(f"GitHub {method} {route} returned HTTP {response.status_code}. Check repository access and Contents read/write; do not print the token.")
            return response.json()

        try:
            source = request("GET", "/git/commits/" + STATE["source_commit"])
            tree_entries = []
            for path, data in evidence.items():
                blob = request("POST", "/git/blobs", {"encoding": "base64", "content": base64.b64encode(data).decode("ascii")})
                tree_entries.append({"path": path, "mode": "100644", "type": "blob", "sha": blob["sha"]})
            tree = request("POST", "/git/trees", {"base_tree": source["tree"]["sha"], "tree": tree_entries})
            commit = request("POST", "/git/commits", {
                "message": f"docs: record Sprint 5 GPU validation {RUN_ID} ({STATE['status']})",
                "tree": tree["sha"], "parents": [STATE["source_commit"]],
            })
            request("POST", "/git/refs", {"ref": "refs/heads/" + branch, "sha": commit["sha"]})
        finally:
            api.headers.clear()
    branch_url = f"https://github.com/{REPOSITORY}/tree/{branch}"
    receipt = {"branch": branch, "commit": commit["sha"], "url": branch_url,
               "tested_source_commit": STATE["source_commit"], "validation_status": STATE["status"]}
    (OUTPUT / "upload-receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
    print("Results branch:", branch_url)
    print("Review before merging. The main branch and GPU environment status were not changed.")
    package_evidence()

if UPLOAD_TO_GITHUB:
    upload_results()
else:
    print("Upload disabled. To publish, enable the Kaggle secret, set UPLOAD_TO_GITHUB=True and rerun this cell.")


## What to send back

Share the results-branch URL printed by cell 7, or upload/download the evidence ZIP
from cell 6. If setup failed before cloning, share the setup log/ZIP. Never share
the token itself. A `PASS` report, its resolved lock and parity/metric evidence
must be reviewed before updating `small-gpu-dev` from `PENDING_HARDWARE`.

References: [Kaggle secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md),
[Kaggle notebooks](https://www.kaggle.com/docs/notebooks),
[uv Python management](https://docs.astral.sh/uv/guides/install-python/),
[GitHub Git database API](https://docs.github.com/en/rest/git/trees).
